In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
# exploring ds
!ls $path/PlantVillage/train

In [ ]:
from torchvision.transforms import Resize, ToTensor, Compose, RandomRotation

In [ ]:
transform = Compose([
    Resize((32, 32)),
    RandomRotation(15),
    ToTensor()
])

In [ ]:
from torch.utils.data import Dataset
import os
from torchvision.datasets import ImageFolder
train_ds = ImageFolder(os.path.join(path, 'PlantVillage', 'train'), transform = transform)
test_ds = ImageFolder(os.path.join(path, 'PlantVillage', 'test'), transform = transform)

In [ ]:
train_ds[0] #checking if ds is ok

In [ ]:
# aok :D

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, 128, shuffle=True)
test_loader = DataLoader(test_ds, 128)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))

# class names
classes = ['early blight', 'healthy', 'late blight']

fig, ax = plt.subplots(1, 6, figsize=(10, 10))

for i in range(6):
    ax[i].axis('off')
    ax[i].imshow(images[i].permute(1, 2, 0).numpy())
    ax[i].set_title(classes[labels[i].item()])

In [ ]:
# Write your code here
import torch.nn as nn

class CNN5(nn.Module):
  def __init__(self):
    super(CNN5, self).__init__()
    self.fwdp = nn.Sequential(
        nn.Conv2d(3, 32, 3, padding='same'), # 32 x 32 x 32
        nn.BatchNorm2d(32), # BONUS: ADDED BATCHNORMMMMMM
        nn.ReLU(),
        nn.MaxPool2d(2, 2), # 32 x 16 x 16
        nn.Conv2d(32, 64, 3, padding='same'), # 64 x 16 x 16
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),# 64 x 8 x 8
        nn.Conv2d(64, 128, 3, padding='same'), # 128 x 8 x 8
        nn.BatchNorm2d(128), # BONUS: ADDED BATCHNORMMMMMM
        nn.ReLU(),
        nn.MaxPool2d(2, 2),# 128 x 4 x 4
        nn.Conv2d(128, 128, 3, padding='same'),# 128 x 4 x 4
        nn.BatchNorm2d(128), # BONUS: ADDED BATCHNORMMMMMM
        nn.ReLU(),
        nn.MaxPool2d(2, 2),# 128 x 2 x 2
        nn.Conv2d(128, 256, 3, padding='same'), # 256 x 2 x 2
        nn.BatchNorm2d(256), # BONUS: ADDED BATCHNORMMMMMM
        nn.Flatten(),
        nn.Linear(256 * 2 * 2, 3) # 3 output classess
    )

  def forward(self, x):
    x = self.fwdp(x)
    return x


In [ ]:
model = CNN5()
model

In [ ]:
from tqdm import tqdm    # Shows progress bar
import torch
# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here
# model already defined
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
display(device)
model.to(device)

learning_rate = 0.0001
num_epochs = 10

optimizer = optim.AdamW(model.parameters(), lr = learning_rate)
loss_fn = nn.CrossEntropyLoss()

tr_losses = []
ts_losses = []
ts_accs = []

In [ ]:
import torch.optim as optim

# comments are cuz copypasted from labs

# Initialize the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()
# i am aware it overfits somewhat like 7 epochs. but ok for this purposes cuz accuracy not important, just task
# shld run successfully.

In [ ]:
# i added concat!!
# Write your code here
import torch.nn as nn

class CNN5_Skip(nn.Module):
  def __init__(self):
    super(CNN5_Skip, self).__init__()
    self.l1to2 = nn.Sequential(
        nn.Conv2d(3, 32, 3, padding='same'), # 32 x 32 x 32 layer1 out
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2, 2), # 32 x 16 x 16
        nn.Conv2d(32, 64, 3, padding='same'), # 64 x 16 x 16 layer2 out
        nn.BatchNorm2d(64),
        nn.ReLU()
    )
    self.l3 = nn.Sequential(
        nn.MaxPool2d(2, 2),# 64 x 8 x 8
        nn.Conv2d(64, 128, 3, padding='same'), # 128 x 8 x 8
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2, 2) # 128 x 4 x 4  # layer3 out
    )
    self.l4onwards = nn.Sequential(
        nn.Conv2d(192, 128, 3, padding='same'),# 192 x 4 x 4 # changed channels cuz 128 from l3, 64 from l2 skipconn. rest same
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),# 128 x 2 x 2
        nn.Conv2d(128, 256, 3, padding='same'), # 256 x 2 x 2
        nn.BatchNorm2d(256),
        nn.Flatten(),
        nn.Linear(256 * 2 * 2, 256),
        nn.Linear(256, 3) # 3 output classess
    )
    self.pool = nn.MaxPool2d(4, 4) # will take l2out from 64 x 4 x 4 -> 64 x 4 x 4, matching with l3out which is 128 x 4 x 4
    # so now total channels for l4 input = 128 + 64 = 192
  def forward(self, x):
    layer2_out = self.l1to2(x)
    layer3_out = self.l3(layer2_out)
    layer2_skip = self.pool(layer2_out)
    x = self.l4onwards(torch.cat([layer2_skip, layer3_out], dim=1))
    return x


In [ ]:
model = CNN5_Skip()
model

In [ ]:
# Write your code here
# model already defined
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
display(device)
model.to(device)

learning_rate = 0.0001
num_epochs = 10

optimizer = optim.AdamW(model.parameters(), lr = learning_rate)
loss_fn = nn.CrossEntropyLoss()

tr_losses = []
ts_losses = []
ts_accs = []

In [ ]:
import torch.optim as optim

# comments are cuz copypasted from labs

# Initialize the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

#reran now again for 10 epochs. see it also fits decent accuracy.
# with skip conns. bonus pls.